## 7) Create Agent

In Section 6, *you* were the loop — the AI flagged a tool call, you ran it, and handed the result back. That works for one call. But what if the question needs two tools? You'd have to build that dispatch logic yourself.

`create_agent` handles that loop for you. Every agent is two things:

**Agent = Model + Harness**
* **Model** — the brain. Reads context and decides: *call a tool, or give the final answer.*
* **Harness** — the loop. Its job is to *get the model the right context at the right time* — run the requested tool, feed the result back, and repeat until the model is done.

![Agent loop](agent_loop.png)

### Code Context
In the code below:
* `create_agent`: Builds the agent — takes a `model`, a list of `tools`, and an optional `system_prompt`.
* `tools=[circle_area, rectangle_area]`: We give the agent **two** tools. It decides which ones to call and when.
* `agent.invoke({"messages": [...]})`: The input is a dict with a `"messages"` key — not a raw list.
* `result["messages"][-1].content`: The agent returns all messages (question → tool calls → tool results → final answer). The last one is the answer.

📚 **[Docs: Agents](https://docs.langchain.com/oss/python/langchain/agents)**

In [2]:
import os
import sys
from pathlib import Path

print(sys.executable)
project_root = Path.cwd()
while project_root != project_root.parent and not (
    project_root / "utils" / "env_utils.py"
).is_file():
    project_root = project_root.parent

if not (project_root / "utils" / "env_utils.py").is_file():
    raise FileNotFoundError("Could not locate utils/env_utils.py from the notebook directory.")

os.chdir(project_root)
sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
from utils.env_utils import doublecheck_env, doublecheck_pkgs

# Load environment variables from .env
load_dotenv()

# Check and print results
doublecheck_env("example.env")  # check environmental variables
doublecheck_pkgs(pyproject_path="pyproject.toml", verbose=True)   # check packages

/Users/tarunkhurana/Engineering/projects/tarunkhurana2015/ai-certification-gen-academy/week1-langchain/.venv/bin/python
Did not find file example.env.
This is used to double check the key settings for the notebook.
This is just a check and is not required.

Python 3.13.5 satisfies requires-python: >=3.11, <3.14
package                | required | installed | status | path                                                                            
---------------------- | -------- | --------- | ------ | --------------------------------------------------------------------------------
jupyter                | >=1.0.0  | 1.1.1     | ✅ OK   | …ai-certification-gen-academy/week1-langchain/.venv/lib/python3.13/site-packages
langchain              | >=1.3.10 | 1.4.5     | ✅ OK   | …ai-certification-gen-academy/week1-langchain/.venv/lib/python3.13/site-packages
langchain-anthropic    | >=1.4.6  | 1.7.6     | ✅ OK   | …ai-certification-gen-academy/week1-langchain/.venv/lib/python3.13/site-packag

In [3]:
from langchain.chat_models import init_chat_model

llm = init_chat_model(model="gpt-4o-mini", temperature=0)


In [5]:
from langchain.agents import create_agent
from langchain.tools import tool
import math

@tool
def circle_area(radius: float) -> float:
    """Return the area of a circle for a given radius.

    Args:
        radius: The radius of the circle
    """
    return math.pi * radius * radius

@tool
def rectangle_area(length: float, width: float) -> float:
    """Return the area of a rectangle given its length and width.

    Args:
        length: The length of the rectangle
        width: The width of the rectangle
    """
    return length * width

agent = create_agent(
    model=llm,
    tools=[circle_area, rectangle_area],
    system_prompt="You are a helpful geometry assistant. Always show your calculations.",
)

In [ ]:
question = "I have a circular pond with radius 3 and a rectangular patio that is 4 by 5. What is the combined area of both?"

result = agent.invoke(
    {
        "messages": [
                {
                    "role": "user", 
                    "content": question
                }
            ]
    }
)

print("Final Answer:", result["messages"][-1].content)


Final Answer: To find the combined area of the circular pond and the rectangular patio, we first calculate the area of each:

1. **Area of the circular pond**:
   \[
   \text{Area} = \pi \times r^2 = \pi \times 3^2 \approx 28.27 \text{ square units}
   \]

2. **Area of the rectangular patio**:
   \[
   \text{Area} = \text{length} \times \text{width} = 4 \times 5 = 20 \text{ square units}
   \]

Now, we add both areas together:
\[
\text{Combined Area} = 28.27 + 20 = 48.27 \text{ square units}
\]

Thus, the combined area of the circular pond and the rectangular patio is approximately **48.27 square units**.


In [7]:
# Peek under the hood — the agent called both tools automatically
print("\n--- Agent's steps ---")
for msg in result["messages"]:
    label = type(msg).__name__
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        print(f"{label}: called → {[tc['name'] for tc in msg.tool_calls]}")
    elif msg.content:
        print(f"{label}: {str(msg.content)[:80]}")


--- Agent's steps ---
HumanMessage: I have a circular pond with radius 3 and a rectangular patio that is 4 by 5. Wha
AIMessage: called → ['circle_area', 'rectangle_area']
ToolMessage: 28.274333882308138
ToolMessage: 20.0
AIMessage: To find the combined area of the circular pond and the rectangular patio, we fir


![Diagram](./images/07_img.png)

In [9]:
## Simple Q/A Chatbot

from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model=llm,
    tools=[circle_area, rectangle_area],
    system_prompt="You are a helpful geometry assistant. Always show your calculations.",
    checkpointer=InMemorySaver(),
)

config = {"configurable": {"thread_id": "my_thread_id"}}

print("Geometry Chatbot ready! Type 'quit' to exit.\n")

while True:
    user_input = input("User: ")
    if not user_input:
        continue
    if user_input.lower() == "quit":
        print("Exiting the chatbot. Goodbye!")
        break
    result = agent.invoke(
        {
            "messages": [
                {
                    "role": "user", 
                    "content": user_input
                }
            ],
        },
        config=config
    )
    print("Bot:", result["messages"][-1].content)


Geometry Chatbot ready! Type 'quit' to exit.

Bot: Hello! How can I assist you today?
Bot: To calculate the area of a circular pond, I need to know the radius of the pond. Could you please provide that information?
Bot: I need the radius of the circular pond to calculate its area. The radius is the distance from the center of the circle to any point on its circumference. If you have that measurement, please share it with me!
Bot: The total area of the circular pond with a radius of 6 units is approximately 113.10 square units. If you have any more questions or need further assistance, feel free to ask!
Exiting the chatbot. Goodbye!
